
<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; color: #000099;">
    <div style="text-align: center;">
        <span style="font-weight: bold; font-size: 18px; letter-spacing: 1px; text-transform: uppercase;">
            Techno-Economic Parameters Data Prossecing
        </span>
        <br>
        <span style="font-weight: bold; font-size: 28px; font-style: italic; letter-spacing: 5px;">
            VPP Features
        </span>
        <br>
        <span style="font-weight: bold; font-size: 18px;">
            Main Formatting Notebook
        </span>
    </div>
    <table style="margin: 20px auto; color: #000099; border-collapse: collapse; text-align: center;">
        <tr style="background-color: #E6E6E6;">
            <td style="font-size: 14px; padding: 10px 40px; opacity: 0.8;"><b>FROM</b></td>
            <td style="font-size: 14px; padding: 10px 40px; opacity: 0.8;"><b>TO</b></td>
        </tr>
        <tr style="background-color: #CCCCCC; font-size: 22px; font-style: italic; font-weight: bold;">
            <td style="padding: 5px 40px 15px 40px;">PYPSA | négaWatt</td>
            <td style="padding: 5px 40px 15px 40px;">DISPA-SET | Unleash</td>
        </tr>
    </table>
    <div style="border-top: 1px solid #000099; padding: 10px">
        This notebook processes techno-economic information extracted from <b>PyPSA</b> simulation outputs and converts the resulting infrastructure, operational, and demand-related datasets into formats compatible with the <b>Dispa-SET Unleash</b> framework.
    </div>
</div>

In [1]:
                            import os
                            import csv
from datetime               import datetime
                            import requests
                            import pandas                      as pd
from shutil                 import move
                            import numpy                       as np
                            import shutil
from bs4                    import BeautifulSoup
                            import re
                            import io
                            import plotly.graph_objects        as go
from typing                 import List, Dict, Tuple
                            import re
from IPython.display        import HTML
from difflib                import get_close_matches
from pathlib                import Path
                            import json
                            import difflib
from rapidfuzz              import process, fuzz
from pprint                 import pprint
                            import shutil

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
    1. Dispa-SET Unleash Folder Path
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
    This step dynamically determines the <b>zone_folder_path</b> by locating the <i>"Dispa-SET_Unleash"</i> directory relative to your current workspace. 
    </div>
</div>

In [2]:
### Get the directory two levels up from the current working directory
dispaSET_unleash_folder_path = Path.cwd().parent.parent
print("dispaSET_unleash_folder_path:", dispaSET_unleash_folder_path)

dispaSET_unleash_folder_path: /home/ray/Dispa-SET_Unleash


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        2. PyPSA Source Scenario
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        There are two primary scenarios available as sources for the PyPSA raw data:<br>
    <span style="color: #000099; margin-right: 10px;"> ● </span> <b> Reference_Scenario   </b>
    <br>
    <span style="color: #000099; margin-right: 10px;"> ● </span> <b> Sufficiency_Scenario </b>
    </div>
</div>

In [3]:
### Set the configuration
# =============================================================================
#pypsa_scenario = "Reference_Scenario"
pypsa_scenario = "Sufficiency_Scenario"
# =============================================================================
print(f"PyPSA Chosen Scenario: {pypsa_scenario}")

PyPSA Chosen Scenario: Sufficiency_Scenario


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
            3. Secondary Folders Path
        </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The internal architecture of the <i>Dispa-SET Unleash</i> directory contains several subfolders that must be mapped to specific path variables for accurate data retrieval.
    </div>
</div>

In [4]:
### Convert the string path into a Path object
dispaSET_unleash_folder_path = Path(dispaSET_unleash_folder_path)
### Base Power Plants Data
power_plants_base_data_folder_path = dispaSET_unleash_folder_path / "Database" / "PowerPlants"
print(f"{'power_plants_base_data_folder_path:':<55} {power_plants_base_data_folder_path}")
print("—" * 140)
### PyPSA Raw Data
power_plants_pypsa_raw_data_folder_path = (
    dispaSET_unleash_folder_path / "RawData_PyPSA" / pypsa_scenario / "PowerPlants"
)
print(f"{'power_plants_pypsa_raw_data_folder_path:':<55} {power_plants_pypsa_raw_data_folder_path}")
print("—" * 140)
### PyPSA Formatted Data
power_plants_pypsa_formated_data_folder_path = (
    dispaSET_unleash_folder_path / "Database_PyPSA" / pypsa_scenario / "PowerPlants"
)
print(f"{'power_plants_pypsa_formated_data_folder_path:':<55} {power_plants_pypsa_formated_data_folder_path}")
print("—" * 140)
### PyPSA Formatted Data with VPP
vpp_scenario_folder = f"{pypsa_scenario}________VPP" 
vpp_power_plants_pypsa_formated_data_folder_path = (
    dispaSET_unleash_folder_path / "Database_PyPSA" / vpp_scenario_folder / "PowerPlants"
)
print(f"{'vpp_power_plants_pypsa_formated_data_folder_path:':<55} {vpp_power_plants_pypsa_formated_data_folder_path}")
print("—" * 140)
'''### Base VPP Power Plants Data File
vpp_power_plants_base_data_folder_path = (
    dispaSET_unleash_folder_path / "Database" / "PowerPlants" / "BE" / "2023_VPP.csv"
)
print(f"{'vpp_power_plants_base_data_folder_path:':<55} {vpp_power_plants_base_data_folder_path}")'''

power_plants_base_data_folder_path:                     /home/ray/Dispa-SET_Unleash/Database/PowerPlants
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
power_plants_pypsa_raw_data_folder_path:                /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Sufficiency_Scenario/PowerPlants
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
power_plants_pypsa_formated_data_folder_path:           /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario/PowerPlants
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
vpp_power_plants_pypsa_formated_data_folder_path:       /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario________VPP/PowerPlants
——————————————————————————————————————————————————————————————————

'### Base VPP Power Plants Data File\nvpp_power_plants_base_data_folder_path = (\n    dispaSET_unleash_folder_path / "Database" / "PowerPlants" / "BE" / "2023_VPP.csv"\n)\nprint(f"{\'vpp_power_plants_base_data_folder_path:\':<55} {vpp_power_plants_base_data_folder_path}")'

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        4. Zone(s) Configuration
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        Define the target geographical zone(s) for data processing. This selection determines which regional datasets will be accessed and where the resulting formatted files will be stored.
        <br>
        Use <b>ISO 3166-1 alpha-2</b> standard codes for European countries (e.g., <i>AT, BE, BG, CH, DE, FR</i>). 
    </div>
</div>

In [5]:
# =============================================================================
all_zones = [
    "AL", "AM", "AT", "AZ", "BY", "BE", "BA", "BG", "HR", "CY", "CZ", "DK",
    "EE", "FI", "FR", "GE", "DE", "EL", "HU", "IS", "IE", "IT", "XK", "LV",
    "LT", "LU", "MT", "MD", "ME", "NL", "MK", "NO", "PL", "PT", "RO", "RU",
    "RS", "SK", "SI", "ES", "SE", "CH", "TR", "UA", "UK"
]
# =============================================================================
active_selection = {"BE", "FR", "DE", "NL", "UK"}
# =============================================================================
### vpp_active_selection = {"BE"}
filter_zones = lambda selected: [z for z in all_zones if z in selected]
zone_names = filter_zones(active_selection)
# vpp_zone_names = filter_zones(vpp_active_selection)
print("Selected Zones:", zone_names)
#print("VPP Selected Zones:", vpp_zone_names)

Selected Zones: ['BE', 'FR', 'DE', 'NL', 'UK']


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: right; color: #000099;">
    <span style="text-transform: uppercase;">
    <b>International Nomenclature</b>
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px">
        The following dictionary maps <b>standardized country codes</b> to their various synonyms, aliases, or alternative naming conventions used across international databases
    <br>
    (e.g., ENTSO-E, Eurostat, or ISO variants).
    <br>
    <b>Purpose:</b> This ensures the script can correctly identify and link data even when source files use inconsistent regional identifiers (e.g., <i>UK</i> vs. <i>GB</i> or <i>EL</i> vs. <i>GR</i>).
    </div>
</div>

In [6]:
### Define a clean base dictionary (no messy padding or unnecessary lists)
# =============================================================================
raw_countries = {
    "AL": ("Albania",            ""),  "AM": ("Armenia",          ""),  "AT": ("Austria",          ""),
    "AZ": ("Azerbaijan",         ""),  "BY": ("Belarus",          ""),  "BE": ("Belgium",          ""),
    "BA": ("Bosnia and Herz.",   ""),  "BG": ("Bulgaria",         ""),  "HR": ("Croatia",          ""),
    "CY": ("Cyprus",             ""),  "CZ": ("Czech Republic",   ""),  "DK": ("Denmark",          ""),
    "EE": ("Estonia",            ""),  "FI": ("Finland",          ""),  "FR": ("France",           ""),
    "GE": ("Georgia",            ""),  "DE": ("Germany",          ""),  "EL": ("Greece",         "GR"),
    "HU": ("Hungary",            ""),  "IS": ("Iceland",          ""),  "IE": ("Ireland",          ""),
    "IT": ("Italy",              ""),  "XK": ("Kosovo",           ""),  "LV": ("Latvia",           ""),
    "LT": ("Lithuania",          ""),  "LU": ("Luxembourg",       ""),  "MT": ("Malta",            ""),
    "MD": ("Moldova",            ""),  "ME": ("Montenegro",       ""),  "NL": ("Netherlands",      ""),
    "MK": ("North Macedonia",    ""),  "NO": ("Norway",           ""),  "PL": ("Poland",           ""),
    "PT": ("Portugal",           ""),  "RO": ("Romania",          ""),  "RU": ("Russia",           ""),
    "RS": ("Serbia",             ""),  "SK": ("Slovakia",         ""),  "SI": ("Slovenia",         ""),
    "ES": ("Spain",              ""),  "SE": ("Sweden",           ""),  "CH": ("Switzerland",      ""),
    "TR": ("Turkey",             ""),  "UA": ("Ukraine",          ""),  "UK": ("United Kingdom", "GB")
}
# =============================================================================
### Constructing dicctionary
zone_names_equivalences_dict = {
    code: {"Acronym": [acronym if acronym else " "], "name": [name]}
    for code, (name, acronym) in raw_countries.items()
}
### Filter the dictionary using only the keys present in zone_names
selected_zone_names_equivalences_dict = {
    k: zone_names_equivalences_dict[k] 
    for k in zone_names 
    if k in zone_names_equivalences_dict
}
### Print only the zones that have a valid alternative acronym
for key, value in selected_zone_names_equivalences_dict.items():
    acronym = value["Acronym"][0].strip()
    if acronym:  # Evaluates to True only if the acronym is not empty or spaces
        print(f"Key: {key};    Acronym: {value['Acronym']};    Name: {value['name']}\n")

Key: UK;    Acronym: ['GB'];    Name: ['United Kingdom']



<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        5. Data Reference Year
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
            Define the <b>target temporal scope</b> for the processing workflow. This variable determines which yearly dataset will be filtered, formatted, and exported.
    </div>
</div>

In [7]:
### Year to which data is formatting to
# =============================================================================
data_target_year = "2030"
#data_target_year = "2040"
#data_target_year = "2050"
# =============================================================================
print(f"Selected Year: {data_target_year}")

Selected Year: 2030


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 12px; text-align: right; color: #000099;">
    <span style="text-transform: uppercase;">
    <b>Tracking & Validation Variables</b>
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px">
        These cells serve to <b>verify and audit</b> the current configuration, confirming file names, directory paths, and scenario-specific metadata.
    <br>
    <span style="font-style: italic; opacity: 0.8;">
        By maintaining these variables, the notebook ensures consistent data flow to subsequent modules without the need for redundant manual entries.
    </span>
    </div>
</div>

In [8]:
### Print the relevant information with automatic 55-character left alignment
print(f"{'Path to the DispaSET Unleash folder:':<55} {dispaSET_unleash_folder_path}")
print("—" * 140)
print(f"{'Path to the Power Plants Base data folder:':<55} {power_plants_base_data_folder_path}")
print("—" * 140)
print(f"{'Path to the Power Plants_Pypsa Raw data folder:':<55} {power_plants_pypsa_raw_data_folder_path}")
print("—" * 140)
print(f"{'Path to the Power Plants_Pypsa Formated data folder:':<55} {power_plants_pypsa_formated_data_folder_path}")
print("—" * 140)
print(f"{'Name of the zones:':<55} {zone_names}")
print("—" * 140)
print(f"{'Name of the zone_names_equivalences_dictionary:':<55} {[v['Acronym'] for v in selected_zone_names_equivalences_dict.values()]}")
print("—" * 140)
print(f"{'Target year:':<55} {data_target_year}")

Path to the DispaSET Unleash folder:                    /home/ray/Dispa-SET_Unleash
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
Path to the Power Plants Base data folder:              /home/ray/Dispa-SET_Unleash/Database/PowerPlants
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
Path to the Power Plants_Pypsa Raw data folder:         /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Sufficiency_Scenario/PowerPlants
————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
Path to the Power Plants_Pypsa Formated data folder:    /home/ray/Dispa-SET_Unleash/Database_PyPSA/Sufficiency_Scenario/PowerPlants
—————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
        6. Raw Data Uploading
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        The nodal capacities generated by the <b>PyPSA</b> optimization outputs are extracted and converted into structured data frames. This transformation is essential for the subsequent alignment of generation limits with the Dispa-SET technical parameters.
        <br>
        <b>Computational Workflow:</b><br> This stage involves parsing the <b>network.generators</b>, <b>network.links</b> and <b>network.storage_units</b> attributes to ensure that capacity values ($P_{nom}$) are correctly localized to their respective nodes.
        <br>
        By converting raw PyPSA outputs into managed data frames, the system enables efficient filtering, scaling, and reformatting of the energy infrastructure data before it is ingested by the Dispa-SET simulation engine.
    </div>
</div>

In [9]:
### Initialize dictionaries to store raw data for generators, links, and storage units for each zone
generators_pypsa_raw_data_dictionary = {}
links_pypsa_raw_data_dictionary = {}
storage_units_pypsa_raw_data_dictionary = {}
### Load CSV files for each zone into the respective dictionaries
for zone in zone_names:
    ### Construct the base path for the current zone and target year
    base_path = (
        power_plants_pypsa_raw_data_folder_path
        / str(data_target_year)
        / zone
    )
    ### Read and store generators, links, and storage units data for the current zone
    generators_pypsa_raw_data_dictionary[zone] = pd.read_csv(
        base_path / f"{zone}_generators.csv"
    )
    links_pypsa_raw_data_dictionary[zone] = pd.read_csv(
        base_path / f"{zone}_links.csv"
    )
    storage_units_pypsa_raw_data_dictionary[zone] = pd.read_csv(
        base_path / f"{zone}_storage_units.csv"
    )
### Print the shape (rows, columns) of each DataFrame for every zone
for zone in zone_names:
    print(
        f"{zone} | "
        f"Generators: {generators_pypsa_raw_data_dictionary[zone].shape} | "
        f"Links: {links_pypsa_raw_data_dictionary[zone].shape} | "
        f"Storage Units: {storage_units_pypsa_raw_data_dictionary[zone].shape}"
    )

BE | Generators: (56, 43) | Links: (100, 61) | Storage Units: (2, 40)
FR | Generators: (54, 43) | Links: (118, 61) | Storage Units: (2, 40)
DE | Generators: (59, 43) | Links: (134, 61) | Storage Units: (2, 40)
NL | Generators: (58, 43) | Links: (101, 61) | Storage Units: (0, 40)
UK | Generators: (115, 43) | Links: (203, 61) | Storage Units: (2, 40)


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       7. Combining PyPSA Raw Data Components
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process combines the three PyPSA component DataFrames (<b>Generators</b>, <b>Links</b>, and <b>Storage Units</b>) into a <b>single unified DataFrame</b> per zone. A new column (<b>PyPSA_Component</b>) tags the origin of each row for filtering and identification.
        <br>
    <b>Combination Logic:</b><br>
        For each zone, the script retrieves the three component DataFrames, adds a <b>PyPSA_Component</b> column with values <b>"Generator"</b>, <b>"Link"</b>, or <b>"StorageUnit"</b>, and then concatenates them vertically using <b>pd.concat()</b>. The resulting DataFrame contains all power units from all three component types.
    </div>
</div>

In [10]:
### Dictionary to store the combined DataFrame per zone
country_dataframes_long = {}
for zone in zone_names:
    ### Retrieve each component DataFrame for the zone
    gen_df = generators_pypsa_raw_data_dictionary[zone].copy()
    link_df = links_pypsa_raw_data_dictionary[zone].copy()
    sto_df = storage_units_pypsa_raw_data_dictionary[zone].copy()
    ### Tag component origin so you can filter/identify rows later
    gen_df["PyPSA_Component"] = "Generator"
    link_df["PyPSA_Component"] = "Link"
    sto_df["PyPSA_Component"] = "StorageUnit"
    ### Concatenate all three components vertically into one combined DataFrame
    combined_df = pd.concat([gen_df, link_df, sto_df], axis=0, ignore_index=True)
    ### Store in the nested/combined dictionary under the country/zone key
    country_dataframes_long[zone] = combined_df
### Console verification report
print("=" * 80)
print("COMBINED PYPSA RAW DATA DICTIONARY SUMMARY")
print("=" * 80)
for zone, df in country_dataframes_long.items():
    gen_count = (df["PyPSA_Component"] == "Generator").sum()
    link_count = (df["PyPSA_Component"] == "Link").sum()
    sto_count = (df["PyPSA_Component"] == "StorageUnit").sum()
    
    print(
        f"Zone: {zone:4s} | Total Rows: {len(df):4d} (Generators: {gen_count:3d}, "
        f"Links: {link_count:3d}, Storage: {sto_count:3d}) | Total Columns: {df.shape[1]}"
    )

COMBINED PYPSA RAW DATA DICTIONARY SUMMARY
Zone: BE   | Total Rows:  158 (Generators:  56, Links: 100, Storage:   2) | Total Columns: 83
Zone: FR   | Total Rows:  174 (Generators:  54, Links: 118, Storage:   2) | Total Columns: 83
Zone: DE   | Total Rows:  195 (Generators:  59, Links: 134, Storage:   2) | Total Columns: 83
Zone: NL   | Total Rows:  159 (Generators:  58, Links: 101, Storage:   0) | Total Columns: 83
Zone: UK   | Total Rows:  320 (Generators: 115, Links: 203, Storage:   2) | Total Columns: 83


/tmp/ipykernel_1680231/2328758735.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat([gen_df, link_df, sto_df], axis=0, ignore_index=True)


<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       8. Filtering Columns in Combined PyPSA Raw Data
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process filters the combined PyPSA DataFrames to retain only the <b>essential columns</b> required for downstream processing. It intelligently handles both column-based and index-based <b>'name'</b> identifiers, ensuring that unit names are preserved regardless of their original location.<br>
    <b>Filtering Logic:</b><br>
        For each country, the script checks which of the target columns exist in the DataFrame and retains only those. It then handles the <b>'name'</b> column separately:
        <br>
        <b>If 'name' is a column</b> → Keeps it as a regular column<br>
        <b>If 'name' is the index</b> → Preserves it as the index and sets <b>index.name = 'name'</b>
    </div>
</div>

In [11]:
### 1. Define the exact columns you want to keep
columns_to_keep = [
    'bus'            , 'bus0'            , 'bus1'                , 'bus2'                 , 'bus3'           , 'bus4'             , 
    'efficiency'     , 'efficiency2'     , 'efficiency3'         , 'efficiency4'          , 'carrier'        , 'start_up_cost'    ,
    'shut_down_cost' , 'stand_by_cost'   , 'min_up_time'         , 'min_down_time'        , 'up_time_before' , 'down_time_before' ,
    'ramp_limit_up'  , 'ramp_limit_down' , 'ramp_limit_start_up' , 'ramp_limit_shut_down' , 'p_nom_opt'      , 		

]

country_dataframes = {}

### 2. Filter the columns of the already built dataframes in-place
for country in country_dataframes_long:
    df = country_dataframes_long[country]
    ### Safely identify columns that exist in the dataframe
    available_cols = [col for col in columns_to_keep if col in df.columns]
    if 'name' in df.columns:
        ### If 'name' is an explicit column, retain it alongside the target features
        country_dataframes[country] = df[['name'] + available_cols]
    else:
        ### If 'name' lives in the index, just filter the features (index is preserved automatically)
        country_dataframes[country] = df[available_cols]
        country_dataframes[country].index.name = 'name'
    print(f"✂️ Cleaned columns for '{country}'. New structure: {country_dataframes[country].shape[1]} columns.")
for country, df in country_dataframes.items():
    print(f"\n{country}")
    print(df.columns.tolist())

✂️ Cleaned columns for 'BE'. New structure: 24 columns.
✂️ Cleaned columns for 'FR'. New structure: 24 columns.
✂️ Cleaned columns for 'DE'. New structure: 24 columns.
✂️ Cleaned columns for 'NL'. New structure: 24 columns.
✂️ Cleaned columns for 'UK'. New structure: 24 columns.

BE
['name', 'bus', 'bus0', 'bus1', 'bus2', 'bus3', 'bus4', 'efficiency', 'efficiency2', 'efficiency3', 'efficiency4', 'carrier', 'start_up_cost', 'shut_down_cost', 'stand_by_cost', 'min_up_time', 'min_down_time', 'up_time_before', 'down_time_before', 'ramp_limit_up', 'ramp_limit_down', 'ramp_limit_start_up', 'ramp_limit_shut_down', 'p_nom_opt']

FR
['name', 'bus', 'bus0', 'bus1', 'bus2', 'bus3', 'bus4', 'efficiency', 'efficiency2', 'efficiency3', 'efficiency4', 'carrier', 'start_up_cost', 'shut_down_cost', 'stand_by_cost', 'min_up_time', 'min_down_time', 'up_time_before', 'down_time_before', 'ramp_limit_up', 'ramp_limit_down', 'ramp_limit_start_up', 'ramp_limit_shut_down', 'p_nom_opt']

DE
['name', 'bus', 'bus

In [12]:
country_dataframes

{'BE':                                                   name    bus  \
 0                                BE1 0 offwind-ac-2030  BE1 0   
 1                                BE1 0 offwind-dc-2030  BE1 0   
 2                                    BE1 0 onwind-2030  BE1 0   
 3                                            BE1 0 ror  BE1 0   
 4                                     BE1 0 solar-2030  BE1 0   
 ..                                                 ...    ...   
 153  BE1 0 services urban decentral resistive heate...    NaN   
 154     BE1 0 services urban decentral gas boiler-2019    NaN   
 155     BE1 0 services urban decentral oil boiler-2019    NaN   
 156                                          BE1 0 PHS  BE1 0   
 157                                        BE1 0 hydro  BE1 0   
 
                   bus0                                 bus1            bus2  \
 0                  NaN                                  NaN             NaN   
 1                  NaN                 

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       9. Heat Load Fraction Allocation
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process allocates <b>capacity fractions</b> for each heat load type across the power units. For each heat load category (e.g., <b>urban central heat</b>), the script identifies all associated technologies, calculates each unit's share of the total capacity, and stores the fraction in a dedicated column.<br>
    <b>Allocation Logic:</b><br>
        The script iterates through all heat load categories and their associated technologies. For each country, it identifies units whose <b>name</b> contains the technology pattern, sums their total <b>p_nom_opt</b> capacity, and calculates the fraction of each unit relative to the total. These fractions are stored in a column named <b>"{heat_load}_fraction"</b>.
    </div>
</div>

In [13]:
### Define all possible heat load options to loop through
# ================================================================
all_heat_loads = [
    'urban central heat',
    'services urban decentral heat',
    'services rural heat',
    'residential rural heat',
    'residential urban decentral heat'
]
# ================================================================
### Process each heat load configuration group sequentially
for current_heat_load in all_heat_loads:
    print(f"\n--- Processing Sector Group: {current_heat_load} ---")
    # 1. Define technologies belonging to the current heat load
    if current_heat_load == 'urban central heat':
        technologies = [
            'urban central resistive heater',
            'urban central air heat pump',
            'urban central solid biomass CHP CC',
            'urban central solid biomass CHP',
            'urban central gas CHP CC',
            'urban central gas CHP',
            'H2 Fuel Cell'
        ]
    elif current_heat_load == 'services urban decentral heat':
        technologies = [
            'services urban decentral resistive heater',
            'services urban decentral air heat pump'
        ]
    elif current_heat_load == 'services rural heat':
        technologies = [
            'services rural resistive heater',
            'services rural air heat pump',
            'services rural ground heat pump'
        ]
    elif current_heat_load == 'residential rural heat':
        technologies = [
            'residential rural resistive heater',
            'residential rural air heat pump',
            'residential rural ground heat pump'
        ]
    elif current_heat_load == 'residential urban decentral heat':

        technologies = [
            'residential urban decentral resistive heater',
            'residential urban decentral air heat pump'
        ]
    else:
        technologies = []
    ### 2. Define the fraction column
    fraction_col = f"{current_heat_load}_fraction"
    ### 3. Create technology pattern
    tech_pattern = "|".join(technologies)
    ### 4. Process each country
    for country, df in country_dataframes.items():
        # Initialize fraction column
        df[fraction_col] = 0.0
        # Find matching technologies in the 'name' column
        mask = (
            df['name']
            .astype(str)
            .str.contains(
                tech_pattern,
                case=False,
                na=False,
                regex=True
            )
        )
        ### Calculate total capacity of matching technologies
        total_tech_p_nom = df.loc[mask, 'p_nom_opt'].sum()
        ### Calculate capacity fractions
        if total_tech_p_nom > 0:
            df.loc[mask, fraction_col] = (
                df.loc[mask, 'p_nom_opt']
                / total_tech_p_nom
            )
            print(
                f"  [{country}] "
                f"Created weights for {mask.sum()} matching assets. "
                f"Total Group Cap: {total_tech_p_nom:.2f} MW"
            )
        else:
            print(
                f"  [{country}] "
                f"No matching assets found."
            )
print("\nAll batch fraction allocations completed successfully!")
### Show resulting columns
for country, df in country_dataframes.items():
    print(f"\n{country}")
    print(df.columns.tolist())


--- Processing Sector Group: urban central heat ---
  [BE] Created weights for 7 matching assets. Total Group Cap: 7815.10 MW
  [FR] Created weights for 7 matching assets. Total Group Cap: 16575.56 MW
  [DE] Created weights for 7 matching assets. Total Group Cap: 46241.72 MW
  [NL] Created weights for 7 matching assets. Total Group Cap: 10691.14 MW
  [UK] Created weights for 14 matching assets. Total Group Cap: 18048.12 MW

--- Processing Sector Group: services urban decentral heat ---
  [BE] Created weights for 4 matching assets. Total Group Cap: 710.42 MW
  [FR] Created weights for 6 matching assets. Total Group Cap: 10293.88 MW
  [DE] Created weights for 4 matching assets. Total Group Cap: 1643.51 MW
  [NL] Created weights for 4 matching assets. Total Group Cap: 2991.33 MW
  [UK] Created weights for 7 matching assets. Total Group Cap: 3880.26 MW

--- Processing Sector Group: services rural heat ---
  [BE] Created weights for 3 matching assets. Total Group Cap: 157.23 MW
  [FR] Crea

/tmp/ipykernel_1680231/3670905064.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[fraction_col] = 0.0
/tmp/ipykernel_1680231/3670905064.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[fraction_col] = 0.0
/tmp/ipykernel_1680231/3670905064.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing

<div style="background-color: #FFFFFF; padding: 15px; font-family: 'Times New Roman', Times, serif; font-size: 14px; text-align: justify; color: #000099;">
    <span style="font-weight: bold;">
       10. Exporting Processed PyPSA Power-Plant DataFrames
    </span>
    <div style="border-top: 1px solid #000099; padding: 10px;">
        This process exports the fully processed PyPSA power-plant DataFrames to <b>CSV files</b>. Each country/zone is saved in a dedicated subfolder organized by year and country, with a descriptive filename for easy identification.<br>
    <b>Export Logic:</b><br>
        For each country, the script creates a subfolder structure: <b>{year}/{country}/</b>. It then saves the DataFrame with the naming convention: <b>pypsa_power_plants_{country}_{year}.csv</b>. The <b>index=False</b> parameter ensures no index column is exported.
    </div>
</div>

In [14]:
### Export processed PyPSA power-plant DataFrames
for country, df in country_dataframes.items():
    ### Define output directory
    output_folder = os.path.join(
        power_plants_pypsa_raw_data_folder_path,
        str(data_target_year),
        country
    )
    ### Create directory if it does not already exist
    os.makedirs(
        output_folder,
        exist_ok=True
    )
    ### Define output filename
    output_filename = (
        f"pypsa_power_plants_{country}_{data_target_year}.csv"
    )
    ### Complete output path
    output_path = os.path.join(
        output_folder,
        output_filename
    )
    ### Export DataFrame
    df.to_csv(
        output_path,
        index=False
    )
    ### Confirmation
    print(
        f"[{country}] Exported {len(df)} rows "
        f"and {len(df.columns)} columns -> {output_path}"
    )
print("\nAll DataFrames exported successfully!")

[BE] Exported 158 rows and 29 columns -> /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Sufficiency_Scenario/PowerPlants/2030/BE/pypsa_power_plants_BE_2030.csv
[FR] Exported 174 rows and 29 columns -> /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Sufficiency_Scenario/PowerPlants/2030/FR/pypsa_power_plants_FR_2030.csv
[DE] Exported 195 rows and 29 columns -> /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Sufficiency_Scenario/PowerPlants/2030/DE/pypsa_power_plants_DE_2030.csv
[NL] Exported 159 rows and 29 columns -> /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Sufficiency_Scenario/PowerPlants/2030/NL/pypsa_power_plants_NL_2030.csv
[UK] Exported 320 rows and 29 columns -> /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Sufficiency_Scenario/PowerPlants/2030/UK/pypsa_power_plants_UK_2030.csv

All DataFrames exported successfully!


In [15]:
country_dataframes['BE'].to_csv('/home/ray/Documents/Unleash/Julios_Data/Auxiliar_Documents/testing_file1.csv', index=True)